# MiniMax H3-Image · Google Colab 运行笔记本

## 模型选型

根据文章《MiniMax H3-Image 正式开源！原生1600万像素，文生图+图片编辑二合一，免费本地部署，效果不错！》，选用 **MiniMax-H3-Image**（HuggingFace: Comfy-Org/MiniMax-H3-Image）。

选型理由：
- 开源权重，免费在 Colab 或本地部署，不依赖官方 API；
- 原生直出最高 1600 万像素，文生图 + 图片编辑二合一，最多 16 张参考图片；
- Colab 免费 T4 显存约 16 GB，匹配文章中 ComfyUI 工作流的 16G 显存版权重。

## 笔记本内容

1. 环境与 GPU 检查
2. 依赖安装（清华大学 pip 镜像）
3. HuggingFace 下载源测速与选源（官方站 vs 镜像站）
4. 下载 MiniMax-H3-Image 权重、VAE、文本编码器（按 16G 显存版）
5. （可选）将模型权重拷贝到 Google Drive
6. 启动 ComfyUI + cloudflared 公网隧道
7. ComfyUI API 调用示例


In [ ]:
import sys
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA 可用:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('显存总量: %.2f GB' % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

!nvidia-smi


In [ ]:
import os
import subprocess
import sys

MIRROR = 'https://pypi.tuna.tsinghua.edu.cn/simple'

def pip_install(*pkgs):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '-i', MIRROR,
           '--trusted-host', 'pypi.tuna.tsinghua.edu.cn'] + list(pkgs)
    print('运行:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

pip_install('huggingface_hub', 'requests', 'tqdm')

# 克隆 ComfyUI
if not os.path.exists('/content/ComfyUI'):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/comfyanonymous/ComfyUI.git',
                    '/content/ComfyUI'], check=True)

pip_install('-r', '/content/ComfyUI/requirements.txt')

# cloudflared（公网隧道）
subprocess.run(['wget', '-q',
                'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
                '-O', '/usr/local/bin/cloudflared'], check=True)
os.chmod('/usr/local/bin/cloudflared', 0o755)
subprocess.run(['which', 'cloudflared'], check=True)


In [ ]:
import os
import time
import requests

REPO = 'Comfy-Org/MiniMax-H3-Image'
CANDIDATES = ['https://huggingface.co', 'https://hf-mirror.com']

results = []
for base in CANDIDATES:
    try:
        t0 = time.time()
        r = requests.head(base + '/api/models/' + REPO, timeout=20, allow_redirects=True)
        dt = time.time() - t0
        results.append((dt, base, r.status_code))
        print('%s  %.2fs  HTTP %s' % (base, dt, r.status_code))
    except Exception as e:
        print('%s  失败: %s' % (base, e))
        results.append((float('inf'), base, None))

results.sort(key=lambda x: x[0])
HF_BASE = results[0][1]
print('\n>>> 选用下载源:', HF_BASE)
os.environ['HF_ENDPOINT'] = HF_BASE.split('//', 1)[1]


In [ ]:
import os
import requests
from huggingface_hub import HfApi, hf_hub_download

REPO = 'Comfy-Org/MiniMax-H3-Image'
HF_API = HfApi(endpoint=os.environ.get('HF_ENDPOINT', 'huggingface.co'))
BASE_DIR = '/content/ComfyUI/models'

def tree(folder):
    for base in (HF_BASE, 'https://huggingface.co'):
        try:
            r = requests.get(base + '/api/models/' + REPO + '/tree/main/' + folder, timeout=30)
            r.raise_for_status()
            return [e for e in r.json() if e.get('type') == 'file']
        except Exception as last:
            print(base, '目录获取失败:', last)
    raise last

def download(remote_path):
    local_path = os.path.join(BASE_DIR, remote_path)
    if os.path.exists(local_path):
        print('跳过已存在:', remote_path)
        return local_path, 0
    print('下载:', remote_path)
    try:
        path = hf_hub_download(repo_id=REPO, filename=remote_path,
                               local_dir=BASE_DIR, local_dir_use_symlinks=False)
    except Exception as e:
        print('镜像站下载失败，回退官方站:', e)
        os.environ.pop('HF_ENDPOINT', None)
        path = hf_hub_download(repo_id=REPO, filename=remote_path,
                               local_dir=BASE_DIR, local_dir_use_symlinks=False)
    size = os.path.getsize(path)
    return path, size

total = 0

# 扩散模型：Colab T4 约 16 GB，优先选带 16 的版本，否则选最大文件
diff = tree('diffusion_models')
diff = [e for e in diff if e['path'].endswith(('.safetensors', '.pt', '.pth', '.bin'))]
c16 = [e for e in diff if '16' in e['path'].lower()]
chosen = c16[0] if c16 else max(diff, key=lambda e: e.get('size', 0))
p, s = download(chosen['path'])
print('扩散模型:', chosen['path'], '%.2f MB' % (s / 1024**2))
total += s

# VAE
for e in tree('vae'):
    if e['path'].endswith(('.safetensors', '.pt', '.pth', '.bin')):
        p, s = download(e['path'])
        print('VAE:', e['path'])
        total += s

# 文本编码器
for e in tree('text_encoders'):
    if e['path'].endswith(('.safetensors', '.pt', '.pth', '.bin')):
        p, s = download(e['path'])
        print('文本编码器:', e['path'])
        total += s

print('\n>>> 下载完成，共 %.2f MB' % (total / 1024**2))


In [ ]:
# 可选：将模型权重拷贝到 Google Drive，避免会话结束后丢失
from google.colab import drive

drive.mount('/content/drive')
dst = '/content/drive/MyDrive/MiniMax-H3-Image'
subprocess.run(['cp', '-r', '/content/ComfyUI/models', dst], check=True)
print('已拷贝到:', dst)


In [ ]:
import os
import re
import subprocess
import time

COMFY = '/content/ComfyUI'
LOG = '/content/comfyui.log'
if os.path.exists(LOG):
    os.remove(LOG)

log_f = open(LOG, 'wb')
proc = subprocess.Popen([sys.executable, 'main.py', '--listen', '--port', '8188'],
                        cwd=COMFY, stdout=log_f, stderr=log_f, start_new_session=True)
time.sleep(10)
print('ComfyUI 已在后台启动，PID:', proc.pid)
if os.path.exists(LOG):
    print(open(LOG, 'r', errors='ignore').read()[-2000:])

# cloudflared 公网隧道
TLOG = '/content/cloudflared.log'
if os.path.exists(TLOG):
    os.remove(TLOG)
tlog_f = open(TLOG, 'wb')
subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8188'],
                 stdout=tlog_f, stderr=tlog_f, start_new_session=True)
time.sleep(12)
text = open(TLOG, 'r', errors='ignore').read() if os.path.exists(TLOG) else ''
m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', text)
TUNNEL_URL = m.group(0) if m else None
if TUNNEL_URL:
    print('\n>>> ComfyUI 公网地址:', TUNNEL_URL)
    print('>>> 在浏览器打开上方地址即可使用 ComfyUI 界面')
else:
    print('\n>>> 未自动获取到隧道地址，请查看 /content/cloudflared.log')
print('ComfyUI API: http://localhost:8188')


In [ ]:
import json
import os
import time
import requests

BASE_URL = TUNNEL_URL or 'http://localhost:8188'

def queue_prompt(workflow):
    r = requests.post(BASE_URL + '/prompt', json={'prompt': workflow}, timeout=60)
    r.raise_for_status()
    return r.json()['prompt_id']

def wait_completion(prompt_id, timeout=300):
    for _ in range(timeout // 3):
        h = requests.get(BASE_URL + '/history/' + prompt_id, timeout=20).json()
        if prompt_id in h:
            return h[prompt_id]
        time.sleep(3)
    raise TimeoutError('生成超时')

wf_path = '/content/workflow_api.json'
if os.path.exists(wf_path):
    workflow = json.load(open(wf_path))
    # 根据你的工作流节点 ID 替换提示词，例如：
    # workflow['6']['inputs']['text'] = 'a beautiful landscape, masterpiece'
    prompt_id = queue_prompt(workflow)
    print('prompt_id:', prompt_id)
    hist = wait_completion(prompt_id)
    print(json.dumps(hist, indent=2)[:3000])
else:
    print('未找到工作流文件:', wf_path)
    print('请先从 https://github.com/vyrekai/minimax-h3-image-comfyui-workflows 下载工作流 JSON，')
    print('上传到 Colab /content 目录，或直接在浏览器打开 ComfyUI 界面拖入工作流运行。')


## 使用说明

1. 依次运行全部单元格（Runtime → Run all），每个单元格执行完毕后再运行下一个。
2. 第 6 个单元格输出的公网地址即为 ComfyUI 界面链接，在浏览器打开后即可拖入工作流、输入提示词生成图片。
3. 工作流文件：文章推荐的工作流可从 https://github.com/vyrekai/minimax-h3-image-comfyui-workflows 下载，Colab T4 显存约 16 GB，请选用 16G 版工作流与对应权重。
4. 模型权重按第 4 个单元格选中的最快 HuggingFace 镜像下载；若镜像站未收录该仓库，会回退到官方站。
5. Colab 免费实例有使用时长限制，断开后需重新运行笔记本；/content 下的文件会话结束后会被清理，如需长期保存请运行第 5 个单元格将权重拷贝到 Google Drive。
